# Train scVI model

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
import os
from pathlib import Path
import scanpy as sc
from scipy.sparse import issparse, csr_matrix
import scvi
import numpy as np
from collections import Counter
import anndata as ad
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
import random
import torch
import sys
import session_info

In [ ]:
random.seed(0)
seed_everything(0)

scvi.settings.seed = 0
scvi.settings.num_workers = 32

## Set paths, import data

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR

input_dir = BASE_DIR / 'data/h5ad/export_01/01b_filter'
scvi_dir = BASE_DIR / 'scvi/models'
output_dir = BASE_DIR / 'data/h5ad/export_02/02a_scvi'

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
path_480   = input_dir / "adata-480.h5ad"
path_475 = input_dir / "adata-475.h5ad"

In [ ]:
adata = sc.read_h5ad(path_475)
bdata = sc.read_h5ad(path_480)

## Prepare adata

In [ ]:
# Save counts layer for scVI and downstream processing
adata.X = adata.layers['counts'].copy()
bdata.X = bdata.layers['counts'].copy()

In [ ]:
for n, a in zip(["adata", "bdata"], [adata, bdata]):
    if not issparse(a.X):
        a.X = csr_matrix(a.X)
        print(f"Converted {n} to sparse CSR")

In [ ]:
adata.X[:5, :5].toarray()

In [ ]:
bdata.X[:5, :5].toarray()

In [ ]:
print('adata.X is sparse:', issparse(adata.X))
print('adata.X has only whole numbers:', np.all(adata.X.data == np.round(adata.X.data)))  # True if all values are whole numbers

# scVI

In [ ]:
def train_scvi_model(adata, model_name, scvi_dir, output_dir, latent_key_suffix=""):
    print(f"\n Training {model_name}")

    # Setup for scVI
    scvi.model.SCVI.setup_anndata(
        adata,
        layer="counts",
        batch_key="slide_id"
    )

    # Initialize model
    model = scvi.model.SCVI(
        adata,
        gene_likelihood="nb"
    )

    # Train model
    model.train(accelerator = 'gpu', early_stopping = True) # updated 12/2

    # Save model
    model_save_dir = os.path.join(scvi_dir, model_name)
    model.save(model_save_dir)

    # Plot ELBO loss curves
    plt.plot(model.history["elbo_train"], label="Train ELBO Loss")
    plt.plot(model.history["elbo_validation"], label="Validation ELBO Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title(f"{model_name} SCVI Training Loss")
    plt.tight_layout()
    plt.show()

    # Add latent representation
    latent_key = f"X_scVI_{model_name}"
    adata.obsm[latent_key] = model.get_latent_representation(adata).astype(np.float32)

    # Save AnnData
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f"{model_name}_adata-scvi.h5ad")
    adata.write_h5ad(filename, compression="gzip")

    print(f"Saved model + latent to {filename} ({latent_key})")

    return model, adata

In [ ]:
train_scvi_model(adata,  "ModelA_475", scvi_dir, output_dir)
train_scvi_model(bdata, "ModelB_480", scvi_dir, output_dir)